# Laboratorio: de OLTP a Data Warehouse

Construiremos un esquema estrella pequeño, validaremos la carga y cerraremos con datos JSON. Funciona en Colab o localmente **sin descargas**; sólo usa la biblioteca estándar de Python.

**Pregunta de negocio:** ¿cómo evoluciona la venta neta por mes, categoría y ciudad?

**Grano:** un renglón de pedido completado.

In [ ]:
import csv
import json
import sqlite3
import tempfile
from datetime import date
from pathlib import Path

def mostrar(conexion, sql, parametros=()):
    cursor = conexion.execute(sql, parametros)
    columnas = [d[0] for d in cursor.description]
    filas = cursor.fetchall()
    anchos = [len(c) for c in columnas]
    for fila in filas:
        anchos = [max(a, len(str(v))) for a, v in zip(anchos, fila)]
    formato = ' | '.join('{:<%d}' % a for a in anchos)
    print(formato.format(*columnas))
    print('-+-'.join('-' * a for a in anchos))
    for fila in filas:
        print(formato.format(*(str(v) for v in fila)))
    return filas

oltp = sqlite3.connect(':memory:')
dw = sqlite3.connect(':memory:')
print('Dos bases listas: OLTP y DW')

## 1. Fuente OLTP

La fuente normalizada registra clientes, productos, pedidos y sus renglones. El precio cobrado queda en el detalle para no reescribir el pasado cuando cambia el precio de lista.

In [ ]:
oltp.executescript('''
CREATE TABLE clientes (
  id_cliente INTEGER PRIMARY KEY, nombre TEXT NOT NULL, ciudad TEXT NOT NULL
);
CREATE TABLE productos (
  id_producto INTEGER PRIMARY KEY, nombre TEXT NOT NULL,
  categoria TEXT NOT NULL, precio_lista REAL NOT NULL
);
CREATE TABLE pedidos (
  id_pedido INTEGER PRIMARY KEY, id_cliente INTEGER NOT NULL,
  fecha TEXT NOT NULL, estado TEXT NOT NULL,
  FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente)
);
CREATE TABLE detalle_pedido (
  id_pedido INTEGER NOT NULL, num_linea INTEGER NOT NULL,
  id_producto INTEGER NOT NULL, cantidad INTEGER NOT NULL,
  precio_unitario REAL NOT NULL, descuento REAL NOT NULL DEFAULT 0,
  PRIMARY KEY (id_pedido, num_linea),
  FOREIGN KEY (id_pedido) REFERENCES pedidos(id_pedido),
  FOREIGN KEY (id_producto) REFERENCES productos(id_producto)
);
''')

oltp.executemany('INSERT INTO clientes VALUES (?, ?, ?)', [
    (1, 'Ana', 'CDMX'), (2, 'Bruno', 'Puebla'), (3, 'Carla', 'CDMX')
])
oltp.executemany('INSERT INTO productos VALUES (?, ?, ?, ?)', [
    (101, 'Café de altura', 'Bebidas', 90.00),
    (102, 'Prensa francesa', 'Equipo', 120.00),
    (103, 'Galletas de avena', 'Alimentos', 55.00),
    (104, 'Té de hierbas', 'Bebidas', 70.00)
])
oltp.executemany('INSERT INTO pedidos VALUES (?, ?, ?, ?)', [
    (1001, 1, '2026-01-15', 'COMPLETADO'),
    (1002, 2, '2026-01-20', 'COMPLETADO'),
    (1003, 1, '2026-01-25', 'CANCELADO'),
    (1004, 3, '2026-02-03', 'COMPLETADO'),
    (1005, 2, '2026-02-10', 'COMPLETADO')
])
oltp.executemany('INSERT INTO detalle_pedido VALUES (?, ?, ?, ?, ?, ?)', [
    (1001, 1, 101, 2, 90.00, 10.00),
    (1001, 2, 102, 1, 120.00, 0.00),
    (1002, 1, 101, 1, 90.00, 0.00),
    (1002, 2, 104, 1, 70.00, 12.50),
    (1003, 1, 102, 1, 120.00, 0.00),
    (1003, 2, 103, 1, 55.00, 0.00),
    (1004, 1, 103, 3, 55.00, 15.00),
    (1005, 1, 103, 1, 55.00, 0.00),
    (1005, 2, 101, 1, 90.00, 30.00)
])
oltp.commit()

mostrar(oltp, '''
SELECT p.id_pedido, p.fecha, p.estado, c.nombre AS cliente,
       d.num_linea, pr.nombre AS producto, d.cantidad,
       d.precio_unitario, d.descuento
FROM pedidos p
JOIN clientes c USING (id_cliente)
JOIN detalle_pedido d USING (id_pedido)
JOIN productos pr USING (id_producto)
ORDER BY p.id_pedido, d.num_linea
''')

## 2. Destino dimensional

Las dimensiones describen **quién, qué y cuándo**. La tabla de hechos registra medidas en el grano declarado.

In [ ]:
dw.executescript('''
PRAGMA foreign_keys = ON;
CREATE TABLE dim_fecha (
  fecha_key INTEGER PRIMARY KEY, fecha TEXT UNIQUE NOT NULL,
  anio INTEGER NOT NULL, mes INTEGER NOT NULL, dia INTEGER NOT NULL
);
CREATE TABLE dim_cliente (
  cliente_key INTEGER PRIMARY KEY AUTOINCREMENT,
  cliente_id_origen INTEGER UNIQUE NOT NULL,
  nombre TEXT NOT NULL, ciudad TEXT NOT NULL
);
CREATE TABLE dim_producto (
  producto_key INTEGER PRIMARY KEY AUTOINCREMENT,
  producto_id_origen INTEGER UNIQUE NOT NULL,
  nombre TEXT NOT NULL, categoria TEXT NOT NULL
);
CREATE TABLE fact_ventas (
  pedido_id_origen INTEGER NOT NULL, num_linea INTEGER NOT NULL,
  fecha_key INTEGER NOT NULL, cliente_key INTEGER NOT NULL,
  producto_key INTEGER NOT NULL, cantidad INTEGER NOT NULL,
  importe_bruto REAL NOT NULL, descuento REAL NOT NULL, venta_neta REAL NOT NULL,
  PRIMARY KEY (pedido_id_origen, num_linea),
  FOREIGN KEY (fecha_key) REFERENCES dim_fecha(fecha_key),
  FOREIGN KEY (cliente_key) REFERENCES dim_cliente(cliente_key),
  FOREIGN KEY (producto_key) REFERENCES dim_producto(producto_key)
);
''')
print('Esquema estrella creado')

## 3. ETL de dimensiones

Conservamos la clave de origen para trazabilidad y generamos claves sustitutas propias del DW.

In [ ]:
clientes = oltp.execute('SELECT id_cliente, nombre, ciudad FROM clientes').fetchall()
productos = oltp.execute('SELECT id_producto, nombre, categoria FROM productos').fetchall()
fechas = [r[0] for r in oltp.execute(
    "SELECT DISTINCT fecha FROM pedidos WHERE estado = 'COMPLETADO'"
).fetchall()]

dw.executemany(
    'INSERT INTO dim_cliente (cliente_id_origen, nombre, ciudad) VALUES (?, ?, ?)',
    clientes
)
dw.executemany(
    'INSERT INTO dim_producto (producto_id_origen, nombre, categoria) VALUES (?, ?, ?)',
    productos
)
for texto in fechas:
    f = date.fromisoformat(texto)
    dw.execute('INSERT INTO dim_fecha VALUES (?, ?, ?, ?, ?)',
               (int(f.strftime('%Y%m%d')), texto, f.year, f.month, f.day))
dw.commit()

cliente_keys = dict(dw.execute(
    'SELECT cliente_id_origen, cliente_key FROM dim_cliente'
).fetchall())
producto_keys = dict(dw.execute(
    'SELECT producto_id_origen, producto_key FROM dim_producto'
).fetchall())
print('cliente_keys:', cliente_keys)
print('producto_keys:', producto_keys)

## 4. ETL de hechos

La regla de negocio excluye pedidos cancelados. Transformamos claves y calculamos medidas antes de cargar.

In [ ]:
filas = oltp.execute('''
SELECT p.id_pedido, d.num_linea, p.fecha, p.id_cliente,
       d.id_producto, d.cantidad, d.precio_unitario, d.descuento
FROM pedidos p
JOIN detalle_pedido d USING (id_pedido)
WHERE p.estado = 'COMPLETADO'
ORDER BY p.id_pedido, d.num_linea
''').fetchall()

for id_pedido, linea, fecha, id_cliente, id_producto, cantidad, precio, descuento in filas:
    bruto = round(cantidad * precio, 2)
    neto = round(bruto - descuento, 2)
    dw.execute('INSERT INTO fact_ventas VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)', (
        id_pedido, linea, int(fecha.replace('-', '')),
        cliente_keys[id_cliente], producto_keys[id_producto],
        cantidad, bruto, descuento, neto
    ))
dw.commit()
print(f'{len(filas)} renglones completados cargados')
mostrar(dw, 'SELECT * FROM fact_ventas ORDER BY pedido_id_origen, num_linea')

## 5. Contratos de calidad

Una ejecución sin excepciones no demuestra que la métrica sea correcta. Conciliamos cantidad, importe y relaciones.

In [ ]:
filas_fuente = oltp.execute('''
SELECT COUNT(*) FROM pedidos p JOIN detalle_pedido d USING (id_pedido)
WHERE p.estado = 'COMPLETADO'
''').fetchone()[0]
filas_hecho = dw.execute('SELECT COUNT(*) FROM fact_ventas').fetchone()[0]
neto_fuente = oltp.execute('''
SELECT SUM(d.cantidad * d.precio_unitario - d.descuento)
FROM pedidos p JOIN detalle_pedido d USING (id_pedido)
WHERE p.estado = 'COMPLETADO'
''').fetchone()[0]
neto_dw = dw.execute('SELECT SUM(venta_neta) FROM fact_ventas').fetchone()[0]
claves_huerfanas = dw.execute('PRAGMA foreign_key_check').fetchall()

assert filas_fuente == filas_hecho == 7
assert round(neto_fuente, 2) == round(neto_dw, 2) == 702.50
assert len(claves_huerfanas) == 0
print('✓ 7 renglones conciliados')
print('✓ Venta neta conciliada: $702.50')
print('✓ 0 claves huérfanas')

## 6. Consulta OLAP

Hacemos *roll-up* por mes y categoría. Después prueba un *drill-down* agregando `c.ciudad` al `SELECT` y al `GROUP BY`.

In [ ]:
consulta_olap = '''
SELECT f.anio, f.mes, p.categoria,
       SUM(v.cantidad) AS unidades,
       ROUND(SUM(v.venta_neta), 2) AS venta_neta
FROM fact_ventas v
JOIN dim_fecha f USING (fecha_key)
JOIN dim_producto p USING (producto_key)
JOIN dim_cliente c USING (cliente_key)
GROUP BY f.anio, f.mes, p.categoria
ORDER BY f.anio, f.mes, p.categoria
'''
mostrar(dw, consulta_olap)

## 7. Cuando llegan datos no tabulares

Dos eventos del mismo flujo pueden tener estructuras distintas. Preservaremos el original en una carpeta conceptual `bronze` y tabularemos sólo los atributos acordados en `silver`. Esto **no implementa** Delta/Iceberg/Hudi; prepara la pregunta para la siguiente clase.

In [ ]:
eventos = [
    {
        'evento': 'ver_producto', 'ts': '2026-02-03T10:15:00', 'cliente_id': 2,
        'producto': {'id': 103, 'categoria': 'Alimentos'},
        'dispositivo': {'tipo': 'movil', 'so': 'Android'}
    },
    {
        'evento': 'agregar_carrito', 'ts': '2026-02-03T10:16:04', 'cliente_id': 2,
        'producto': {'id': 103, 'categoria': 'Alimentos'},
        'dispositivo': {'tipo': 'movil', 'so': 'Android'},
        'experimento': {'nombre': 'checkout_v2', 'variante': 'B'}
    }
]
print('Llaves del evento 1:', sorted(eventos[0]))
print('Llaves del evento 2:', sorted(eventos[1]))

raiz = Path(tempfile.mkdtemp(prefix='lakehouse_demo_'))
bronze = raiz / 'bronze'
silver = raiz / 'silver'
bronze.mkdir(); silver.mkdir()

with (bronze / 'clickstream.jsonl').open('w', encoding='utf-8') as f:
    for evento in eventos:
        f.write(json.dumps(evento, ensure_ascii=False) + '\n')

with (silver / 'eventos.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'evento', 'ts', 'cliente_id', 'producto_id', 'dispositivo_tipo'
    ])
    writer.writeheader()
    for e in eventos:
        writer.writerow({
            'evento': e['evento'], 'ts': e['ts'], 'cliente_id': e['cliente_id'],
            'producto_id': e['producto']['id'],
            'dispositivo_tipo': e['dispositivo']['tipo']
        })

print('Bronze conserva:', bronze / 'clickstream.jsonl')
print('Silver publica:', silver / 'eventos.csv')
print('\nLa próxima pregunta: ¿cómo agregamos catálogo, esquema, versiones y transacciones')
print('para administrar muchos archivos como tablas confiables?')

## Cierre

1. El DW separó la operación de la analítica.
2. El grano protegió el significado de cada fila.
3. Las pruebas conciliaron fuente y destino.
4. Los datos diversos no obligan por sí solos a usar lakehouse; abren esa evaluación cuando se combinan escala, evolución, conservación y múltiples cargas de trabajo.